# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 11 — Chuẩn bị khảo sát thực nghiệm trọng số trên BAF

    Mục đích: tạo protocol/cache/nguồn tác giả dùng chung. Bước này chưa huấn luyện cải tiến đề xuất. Đọc `docs/RESEARCH_METHODS.md` trước khi Run All. Các notebook12–15 lưu training thật trong notebook con, notebook16 mở test khi mọi suite đã chọn đã khóa.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


## 1. Chọn phạm vi trước khi tạo campaign

    Mặc định full BAF, ba seed, có LSR. LSR cần VAE + hai diffusion model và hàng trăm triệu lượt denoising; streaming giảm RAM/disk nhưng vẫn tốn thời gian. Nếu ngân sách chưa đủ, đặt `INCLUDE_LSR=False` trước khi chạy. Kết quả pilot chỉ kiểm kỹ thuật. Không sửa cấu hình khi resume campaign.

In [2]:

DATA_PATH = Path('/workspace/scratch/a3538503433e/work/research11/synthetic_validation/SYNTHETIC_NOT_BAF.csv')
INCLUDE_LSR = True
RESUME_CAMPAIGN = None
CONFIG = gru.ResearchConfig(
    mode="synthetic",
    seeds=(42, 317),
    threads=1,
    device="cpu",
    meta_per_class=4,
    epochs=2,
    batch_size=64,
    meta_batch_size=32,
    hidden=(16, 8),
    learning_rates=(.03,),
    patience=2,
    tree_rounds=10,
    vae_epochs=2,
    diffusion_epochs=2,
    auxiliary_patience=2,
    auxiliary_batch_size=64,
    diffusion_width=32,
    score_batch_size=128,
    score_noise_repeats=2,
    score_steps=3,
    score_noise_chunk=1,
    bootstrap_repeats=30,
    pilot_train=1000,
    pilot_eval=200,
)
DATASET_LABEL = "SYNTHETIC_NOT_BAF_960_ROWS"
SUITES = ["losses", "meta", "lsr", "xgboost"]
print("SYNTHETIC TECHNICAL VALIDATION ONLY; THESE ARE NOT BAF RESULTS")


SYNTHETIC TECHNICAL VALIDATION ONLY; THESE ARE NOT BAF RESULTS


## 2. Clone và kiểm nguồn; fit preprocessing chỉ trên core train

    Kiểm hash Base.csv, commit và file nguồn tác giả. Giữ meta cân bằng trong train tháng5; loại các hàng meta khỏi train của mọi learner. Validation tháng6, test tháng7. Chỉ preprocessing được áp dụng lên test ở đây; chưa chọn model hoặc tính test metric.

In [3]:
CAMPAIGN = gru.research_create_campaign(
    project=ROOT,
    data_path=DATA_PATH,
    config=CONFIG,
    resume=RESUME_CAMPAIGN,
    dataset_label=DATASET_LABEL,
    suites=SUITES,
)
print("Lưu đường dẫn này để dùng ở 12–16:", CAMPAIGN)


SOURCE_VERIFIED cb 1d7857208a2abc03d84e35a9d5383af8225d4b4d
SOURCE_VERIFIED ldam 2536330f2afdaa65618323cb5a5850efccce762a
SOURCE_VERIFIED lre 0b616c99ecf8a1c99925322167694272a966ed00
SOURCE_VERIFIED mwn 20c3276b9bf2a192dbaba17cdaef1fd280338d7f
SOURCE_VERIFIED lsr f8561071b72942e0001612bfea505aaeadbb4e73
SOURCE_VERIFIED imxgb 6d82dc83266a32c8579a3e8ea172378085ad1711
              n fraud              months
train       712   104  [0, 1, 2, 3, 4, 5]
validation  120    18                 [6]
test        120    18                 [7]
meta          8     4                 [5]
RESEARCH_DATA_READY /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_data/2b26395e7424dc50
CAMPAIGN_READY /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Lưu đường dẫn này để dùng ở 12–16: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weightin

## 3. Kiểm thông tin đã khóa

    Đây là manifest thực tế để đối chiếu số hàng, split và source. Sau cell này chạy12,13,14 (nếu chọn) và15 lần lượt. Giữ nguyên campaign; chưa chạy16 khi còn thiếu suite.

In [4]:
import pandas as pd

plan = gru.research_read(CAMPAIGN / "campaign.json")
data = gru.research_read(Path(plan["cache"]) / "metadata.json")
display(pd.DataFrame(data["counts"]).T)
print("Data SHA256:", data["data_sha256"])
print("Campaign file:", CAMPAIGN / "campaign.json")
print("Mode:", data["mode"])


Data SHA256: 009fcec4e3ac82dbb08e9a32e7cf225f72c27c7f889c67629b9b6f242d86faf8
Campaign file: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/campaign.json
Mode: synthetic


,n,fraud,months
train,712,104,"[0, 1, 2, 3, 4, 5]"
validation,120,18,[6]
test,120,18,[7]
meta,8,4,[5]
